In [1]:
import scipy as sp
import numpy as np
from plotly import graph_objects as go
from bayes_changepoint.stats import NormalInverseGamma, discrete_distribution
from bayes_changepoint.model import RunLengthModel

In [2]:
# data simulation
# for now I choose some simple model with two states (two distinct sets of distribution parameters) 
# where the switching process will be modelled as a discrete markov chain
n_state = 2
transition_dist: list[discrete_distribution] = [discrete_distribution([0, 1], [0.99, 0.01]), 
                   discrete_distribution([0, 1], [0.01, 0.99])]
init_dist = discrete_distribution([0, 1], [0.5, 0.5])

# arbitrary values of the hyperparameters
mu_prior = 0.0
n_prior = 1
nu_prior = 10
sigma_prior = 2

variance_prior = sp.stats.invgamma(a=nu_prior/2, scale=nu_prior*sigma_prior**2/2)
state_distributions = []

for i_state in range(n_state):
    variance = variance_prior.rvs()
    print(variance)
    mean_prior = sp.stats.norm(loc=mu_prior, scale=np.sqrt(variance)/n_prior)
    mean = mean_prior.rvs()
    print(mean)
    state_distributions.append(sp.stats.norm(loc=mean, scale=np.sqrt(variance)))

n_step = 1000
chain = []
chain.append(init_dist.sample())
samples = []
samples.append(state_distributions[chain[0]].rvs())

for i_step in range(1, n_step):
    chain.append(transition_dist[chain[i_step-1]].sample())
    samples.append(state_distributions[chain[i_step]].rvs())

fig_obj = go.Figure()
fig_obj.add_trace(go.Scatter(y=chain, mode="markers+lines", name="True state"))
fig_obj.show()

fig_obj = go.Figure()
fig_obj.add_trace(go.Scatter(y=samples, mode="markers+lines", name="Samples"))
fig_obj.show()


5.335848781914639
1.458530824919671
8.147424660309436
0.3442154927947209


In [3]:
# We start with the assumption of no history, i.e. only run length of zero is assumed.
run_length_model = RunLengthModel([0], 
                                  [NormalInverseGamma(mu_prior, sigma_prior, n_prior, mu_prior)],
                                  [1.0])

for i_step in range(n_step):
    likelihoods = run_length_model.likelihood(samples[i_step])
    print(likelihoods)


[np.float64(0.055770802811447424)]
[np.float64(0.20930916975429287)]
[np.float64(0.03347329774922669)]
[np.float64(0.037466934033485146)]
[np.float64(0.1621094512797814)]
[np.float64(0.07038421401547505)]
[np.float64(0.08062487016426689)]
[np.float64(0.08161711586022753)]
[np.float64(0.07656097240162589)]
[np.float64(0.15816028781492641)]
[np.float64(0.16820209134420064)]
[np.float64(0.13166556677053576)]
[np.float64(0.03803103774675697)]
[np.float64(0.18846021218899534)]
[np.float64(0.013950804387644124)]
[np.float64(0.162381027512718)]
[np.float64(0.2134490727831579)]
[np.float64(0.021296988919518353)]
[np.float64(0.04505188277717124)]
[np.float64(0.03823051840227312)]
[np.float64(0.10692658339599269)]
[np.float64(0.06519295900959189)]
[np.float64(0.15545029356798748)]
[np.float64(0.03742702532534418)]
[np.float64(0.1656638025860016)]
[np.float64(0.06988265153708181)]
[np.float64(0.0863684827345891)]
[np.float64(0.034787569693642106)]
[np.float64(0.05813496002540508)]
[np.float64(0.2